# Vectorizors:

1. One-Hot vectorizes by making a set of the vocabulary and placing a 1 if the word is present and 0 if not. For example:
sentence\vocab | play | I | am | playing | grass | 
I am playing   |  0   | 1 |  1 |    1    |   0   |

so the vector is 01110 for the sentence and for a word (which is the primary use for One-Hot encoder) it is only the word itself as a one and the rest is 0

2. The count vectorizer works in the same exact way it just places the count of a term in the sentence instead of and 0 if it is not there.

3. TF-IDF is the same aswell but instead of having counts it normalizes the count and then calculates the IDF and finally uses the TF-IDF values to make the vector for a document or a sentence and zeros can occur but you can remove them by applying smoothing which just makes it IDF = log(N+1/DF(t)+1​)+1 but if the TF is 0 the result is 0.

In [6]:
from collections import Counter
import math
import re

import pandas as pd


def tokenize(text):
    """Lowercase text and return its word tokens."""
    if not isinstance(text, str):
        raise TypeError("Each document must be a string.")
    return re.findall(r"\b\w+\b", text.lower())


def prepare_documents(documents, vocabulary=None):
    """Validate, tokenize, and create or reuse a fixed vocabulary."""
    if isinstance(documents, str):
        raise TypeError("documents must be a collection of strings.")

    documents = list(documents)
    if not documents:
        raise ValueError("At least one document is required.")

    tokenized_documents = [tokenize(document) for document in documents]

    if vocabulary is None:
        vocabulary = sorted(
            {token for document in tokenized_documents for token in document}
        )
    else:
        vocabulary = list(vocabulary)
        if len(vocabulary) != len(set(vocabulary)):
            raise ValueError("The vocabulary cannot contain duplicate terms.")

    return tokenized_documents, vocabulary


sample_documents = [
    "Machine learning uses data",
    "Deep learning uses neural networks",
    "Data data powers learning",
]

In [7]:
def one_hot_vectorizer(documents, vocabulary=None):
    tokenized_documents, vocabulary = prepare_documents(documents, vocabulary)

    vectors = []
    for document_tokens in tokenized_documents:
        terms_in_document = set(document_tokens)
        vector = [int(term in terms_in_document) for term in vocabulary]
        vectors.append(vector)

    return vectors, vocabulary


one_hot_vectors, vocabulary = one_hot_vectorizer(sample_documents)
one_hot_results = pd.DataFrame(
    one_hot_vectors, columns=vocabulary, index=["document_1", "document_2", "document_3"]
)

assert one_hot_results.loc["document_1", "machine"] == 1
assert one_hot_results.loc["document_2", "machine"] == 0
assert one_hot_results.loc["document_3", "data"] == 1
one_hot_results

,data,deep,learning,machine,networks,neural,powers,uses
document_1,1,0,1,1,0,0,0,1
document_2,0,1,1,0,1,1,0,1
document_3,1,0,1,0,0,0,1,0


In [8]:
def count_vectorizer(documents, vocabulary=None):
    """Return term-count document vectors and their vocabulary."""
    tokenized_documents, vocabulary = prepare_documents(documents, vocabulary)

    vectors = []
    for document_tokens in tokenized_documents:
        term_counts = Counter(document_tokens)
        vector = [term_counts.get(term, 0) for term in vocabulary]
        vectors.append(vector)

    return vectors, vocabulary


count_vectors, count_vocabulary = count_vectorizer(
    sample_documents, vocabulary=vocabulary
)
count_results = pd.DataFrame(
    count_vectors,
    columns=count_vocabulary,
    index=["document_1", "document_2", "document_3"],
)

assert count_results.loc["document_3", "data"] == 2
assert count_results.loc["document_1", "data"] == 1
assert count_results.loc["document_2", "data"] == 0
count_results

,data,deep,learning,machine,networks,neural,powers,uses
document_1,1,0,1,1,0,0,0,1
document_2,0,1,1,0,1,1,0,1
document_3,2,0,1,0,0,0,1,0


In [9]:
def tf_idf_vectorizer(documents, vocabulary=None):
    tokenized_documents, vocabulary = prepare_documents(documents, vocabulary)
    number_of_documents = len(tokenized_documents)

    document_frequencies = {
        term: sum(term in set(document) for document in tokenized_documents)
        for term in vocabulary
    }
    inverse_document_frequencies = {
        term: math.log(number_of_documents / document_frequencies[term])
        if document_frequencies[term] > 0
        else 0.0
        for term in vocabulary
    }

    vectors = []
    for document_tokens in tokenized_documents:
        term_counts = Counter(document_tokens)
        token_count = len(document_tokens)
        vector = []

        for term in vocabulary:
            term_frequency = (
                term_counts.get(term, 0) / token_count if token_count else 0.0
            )
            vector.append(term_frequency * inverse_document_frequencies[term])

        vectors.append(vector)

    return vectors, vocabulary, inverse_document_frequencies


tf_idf_vectors, tf_idf_vocabulary, idf_values = tf_idf_vectorizer(
    sample_documents, vocabulary=vocabulary
)
tf_idf_results = pd.DataFrame(
    tf_idf_vectors,
    columns=tf_idf_vocabulary,
    index=["document_1", "document_2", "document_3"],
)

# `learning` occurs in every document, so log(N / DF) is log(1) = 0.
assert idf_values["learning"] == 0.0
assert tf_idf_results.loc["document_1", "machine"] > 0.0
assert tf_idf_results.loc["document_2", "machine"] == 0.0
print("TF-IDF document vectors:")
tf_idf_results

TF-IDF document vectors:


,data,deep,learning,machine,networks,neural,powers,uses
document_1,0.101366,0.000000,0.0,0.274653,0.000000,0.000000,0.000000,0.101366
document_2,0.000000,0.219722,0.0,0.000000,0.219722,0.219722,0.000000,0.081093
document_3,0.202733,0.000000,0.0,0.000000,0.000000,0.000000,0.274653,0.000000


In [10]:
# Vectorize every IMDB review while limiting the vocabulary and storing only
# non-zero values. Dense Python lists for 50,000 x 1,000 values per method
# would consume too much memory, so the same vectorization rules are applied
# directly to sparse matrices.
from array import array
from pathlib import Path
import sys

import numpy as np
from scipy.sparse import coo_matrix


current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "phase-1-machine-learning-nlp").exists()
)
part_02 = repository_root / "phase-1-machine-learning-nlp" / "02-preprocessing-tokenization"
sys.path.insert(0, str(part_02 / "src"))
from preprocessor import preprocess_text

imdb_path = part_02 / "data" / "IMDB Dataset.csv"
imdb_reviews = pd.read_csv(imdb_path)
if "review" not in imdb_reviews.columns:
    raise KeyError("The IMDB dataset must contain a 'review' column.")
if imdb_reviews["review"].isna().any():
    raise ValueError("The IMDB review column contains missing values.")

MIN_DOCUMENT_FREQUENCY = 25
MAX_DOCUMENT_FREQUENCY_RATIO = 0.80
MAX_FEATURES = 1_000


def prepare_imdb_tokens(review):
    """Clean a review, remove stop words, and retain useful word tokens."""
    cleaned_review = preprocess_text(
        str(review), remove_stopwords=True, apply_stemming=False
    )
    return [
        token
        for token in cleaned_review.split()
        if token.isalpha() and len(token) >= 2
    ]


# First pass over all reviews: calculate corpus and document frequencies.
corpus_term_counts = Counter()
corpus_document_frequencies = Counter()
for review in imdb_reviews["review"]:
    review_tokens = prepare_imdb_tokens(review)
    corpus_term_counts.update(review_tokens)
    corpus_document_frequencies.update(set(review_tokens))

number_of_documents = len(imdb_reviews)
maximum_document_frequency = int(
    MAX_DOCUMENT_FREQUENCY_RATIO * number_of_documents
)
eligible_terms = [
    term
    for term, frequency in corpus_document_frequencies.items()
    if MIN_DOCUMENT_FREQUENCY <= frequency <= maximum_document_frequency
]

# Keep the most frequent eligible terms; alphabetical ordering resolves ties.
imdb_vocabulary = sorted(
    eligible_terms, key=lambda term: (-corpus_term_counts[term], term)
)[:MAX_FEATURES]
if not imdb_vocabulary:
    raise ValueError("The vocabulary-selection rules removed every token.")
vocabulary_indexes = {term: index for index, term in enumerate(imdb_vocabulary)}

# Second pass: collect each non-zero document-term count compactly.
row_indexes = array("I")
column_indexes = array("I")
nonzero_counts = array("I")
document_lengths = np.zeros(number_of_documents, dtype=np.int32)

for document_index, review in enumerate(imdb_reviews["review"]):
    review_tokens = prepare_imdb_tokens(review)
    document_lengths[document_index] = len(review_tokens)
    selected_counts = Counter(
        token for token in review_tokens if token in vocabulary_indexes
    )

    for term, count in selected_counts.items():
        row_indexes.append(document_index)
        column_indexes.append(vocabulary_indexes[term])
        nonzero_counts.append(count)

row_indexes = np.frombuffer(row_indexes, dtype=np.uint32)
column_indexes = np.frombuffer(column_indexes, dtype=np.uint32)
count_values = np.frombuffer(nonzero_counts, dtype=np.uint32)
matrix_shape = (number_of_documents, len(imdb_vocabulary))

# One-hot stores presence, while count stores the number of occurrences.
imdb_one_hot_matrix = coo_matrix(
    (np.ones(len(count_values), dtype=np.uint8), (row_indexes, column_indexes)),
    shape=matrix_shape,
).tocsr()
imdb_count_matrix = coo_matrix(
    (count_values.astype(np.int32), (row_indexes, column_indexes)),
    shape=matrix_shape,
).tocsr()

# TF-IDF uses normalized TF and IDF = log(N / DF), matching the function above.
idf_values = np.array(
    [
        math.log(number_of_documents / corpus_document_frequencies[term])
        for term in imdb_vocabulary
    ],
    dtype=np.float32,
)
safe_document_lengths = np.maximum(document_lengths[row_indexes], 1)
tf_idf_values = (
    count_values.astype(np.float32) / safe_document_lengths
) * idf_values[column_indexes]
imdb_tf_idf_matrix = coo_matrix(
    (tf_idf_values, (row_indexes, column_indexes)), shape=matrix_shape
).tocsr()

assert imdb_one_hot_matrix.shape == matrix_shape
assert imdb_count_matrix.shape == matrix_shape
assert imdb_tf_idf_matrix.shape == matrix_shape
assert imdb_one_hot_matrix.nnz == imdb_count_matrix.nnz == imdb_tf_idf_matrix.nnz

print(f"IMDB reviews vectorized: {number_of_documents:,}")
print(f"Unique cleaned tokens before selection: {len(corpus_term_counts):,}")
print(f"Selected vocabulary size: {len(imdb_vocabulary):,}")
print(f"Matrix shape: {matrix_shape}")
print(f"Stored non-zero values per matrix: {imdb_count_matrix.nnz:,}")
print(f"First 20 vocabulary terms: {imdb_vocabulary[:20]}")
print("One-hot, count, and TF-IDF matrices were created for every review.")

# Vectorize one readable example with the same vocabulary and corpus IDF.
example_sentence = "This movie has a great story and great acting"
example_tokens = prepare_imdb_tokens(example_sentence)
example_counts = Counter(
    token for token in example_tokens if token in vocabulary_indexes
)

example_one_hot_vector = np.zeros(len(imdb_vocabulary), dtype=np.uint8)
example_count_vector = np.zeros(len(imdb_vocabulary), dtype=np.int32)
example_tf_idf_vector = np.zeros(len(imdb_vocabulary), dtype=np.float32)

for term, count in example_counts.items():
    term_index = vocabulary_indexes[term]
    example_one_hot_vector[term_index] = 1
    example_count_vector[term_index] = count
    term_frequency = count / len(example_tokens)
    example_tf_idf_vector[term_index] = term_frequency * idf_values[term_index]

nonzero_term_indexes = np.flatnonzero(example_count_vector)
example_feature_values = {
    imdb_vocabulary[index]: {
        "index": int(index),
        "one_hot": int(example_one_hot_vector[index]),
        "count": int(example_count_vector[index]),
        "tf_idf": float(example_tf_idf_vector[index]),
    }
    for index in nonzero_term_indexes
}

print("\n" + "=" * 100)
print(f"Example sentence: {example_sentence!r}")
print(f"Processed tokens: {example_tokens}")
print(f"Vector length: {len(imdb_vocabulary)}")
print(f"Non-zero feature values: {example_feature_values}")
print(f"One-hot vector:\n{example_one_hot_vector.tolist()}")
print(f"Count vector:\n{example_count_vector.tolist()}")
print(f"TF-IDF vector:\n{example_tf_idf_vector.tolist()}")

IMDB reviews vectorized: 50,000
Unique cleaned tokens before selection: 214,481
Selected vocabulary size: 1,000
Matrix shape: (50000, 1000)
Stored non-zero values per matrix: 2,634,948
First 20 vocabulary terms: ['movie', 'film', 'one', 'like', 'good', 'even', 'would', 'time', 'really', 'see', 'story', 'much', 'well', 'get', 'great', 'bad', 'also', 'people', 'first', 'dont']
One-hot, count, and TF-IDF matrices were created for every review.

Example sentence: 'This movie has a great story and great acting'
Processed tokens: ['movie', 'great', 'story', 'great', 'acting']
Vector length: 1000
Non-zero feature values: {'movie': {'index': 0, 'one_hot': 1, 'count': 1, 'tf_idf': 0.10249873250722885}, 'story': {'index': 10, 'one_hot': 1, 'count': 1, 'tf_idf': 0.24861204624176025}, 'great': {'index': 14, 'one_hot': 1, 'count': 2, 'tf_idf': 0.5589582920074463}, 'acting': {'index': 35, 'one_hot': 1, 'count': 1, 'tf_idf': 0.31413963437080383}}
One-hot vector:
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0